# KOT-V2-Mini: Russian TinyStories Weight Lab

Интерактивный эксперимент: маленький KOT-подобный Transformer с GQA, RoPE, RMSNorm, SwiGLU и MoE Top-2/4. После обучения ползунок меняет степень расхождения четырёх FFN-экспертов относительно их среднего веса. Каждое изменение пересчитывает validation loss и генерацию.

**Важно:** это учебный мини-эксперимент с символьным токенизатором, а не полноценная KOT-V2-2B. Для эксперимента нужен GPU runtime (T4 подходит). Датасет: `Dxniz/TinyStories-Multilingual`, только строки `language_code == 'ru'`.

In [ ]:
import os, math, copy, random, time, asyncio, subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "ipywidgets", "matplotlib"])
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from IPython.display import display, clear_output
import ipywidgets as widgets

assert torch.cuda.is_available(), 'Включи GPU: Runtime → Change runtime type → T4 GPU'
device = 'cuda'
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
torch.manual_seed(42); np.random.seed(42); random.seed(42)


## 1. Загружаем русские TinyStories

В репозитории датасета есть поле `language_code`; выбираем только `ru`, а текст берём из `output`. Используем символьную токенизацию, чтобы эксперимент был лёгким и не требовал отдельного BPE-токенизатора.

In [ ]:
ds = load_dataset('Dxniz/TinyStories-Multilingual', split='train')
ru = ds.filter(lambda x: x['language_code'] == 'ru')
stories = [s.strip() for s in ru['output'] if isinstance(s, str) and len(s.strip()) >= 80]
print('Русских историй:', len(stories))
if len(stories) < 100:
    raise RuntimeError('Получено слишком мало русских историй. Проверь доступность датасета.')

random.Random(42).shuffle(stories)
split = max(1, int(len(stories) * 0.95))
train_stories, val_stories = stories[:split], stories[split:]
chars = sorted(set(''.join(train_stories + val_stories) + '\n'))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
vocab_size = len(chars)
train_text = '\n'.join(train_stories) + '\n'
val_text = '\n'.join(val_stories) + '\n'
train_ids = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_ids = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)
print(f'Символов в словаре: {vocab_size} | train: {len(train_ids):,} | val: {len(val_ids):,}')
print('\nПример:', train_stories[0][:350])


## 2. Архитектура KOT-V2-Mini

4 слоя, hidden size 192, GQA (6 Q-heads / 2 KV-heads), RoPE, RMSNorm, SwiGLU и MoE из четырёх экспертов с Top-2 routing. Чтобы эксперты не были абсолютно одинаковыми, при старте их копии получают небольшой шум.

In [ ]:
# Быстрые параметры для T4. При желании можно менять до запуска обучения.
BLOCK_SIZE = 256
BATCH_SIZE = 32
MAX_STEPS = 1200
EVAL_EVERY = 100
EVAL_BATCHES = 12
LR = 3e-4
N_LAYER = 4
N_HEAD = 6
N_KV_HEAD = 2
N_EMBD = 192
N_EXPERTS = 4
TOP_K = 2
DROPOUT = 0.05

assert N_EMBD % N_HEAD == 0 and N_HEAD % N_KV_HEAD == 0
HEAD_DIM = N_EMBD // N_HEAD

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype) * self.weight

def rope(x):
    # x: [B, heads, T, head_dim]
    B,H,T,D = x.shape
    half = D // 2
    inv = 1.0 / (10000 ** (torch.arange(0, half, device=x.device, dtype=torch.float32) / half))
    pos = torch.arange(T, device=x.device, dtype=torch.float32)
    ang = torch.outer(pos, inv)
    cos, sin = ang.cos()[None,None,:,:], ang.sin()[None,None,:,:]
    a, b = x.float()[..., :half], x.float()[..., half:2*half]
    y = torch.cat((a*cos - b*sin, a*sin + b*cos), dim=-1)
    if D % 2: y = torch.cat((y, x.float()[..., -1:]), dim=-1)
    return y.to(x.dtype)

class GQAAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.q = nn.Linear(N_EMBD, N_HEAD*HEAD_DIM, bias=False)
        self.k = nn.Linear(N_EMBD, N_KV_HEAD*HEAD_DIM, bias=False)
        self.v = nn.Linear(N_EMBD, N_KV_HEAD*HEAD_DIM, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = nn.Dropout(DROPOUT)
    def forward(self, x):
        B,T,C = x.shape
        q = self.q(x).view(B,T,N_HEAD,HEAD_DIM).transpose(1,2)
        k = self.k(x).view(B,T,N_KV_HEAD,HEAD_DIM).transpose(1,2)
        v = self.v(x).view(B,T,N_KV_HEAD,HEAD_DIM).transpose(1,2)
        q, k = rope(q), rope(k)
        k = k.repeat_interleave(N_HEAD//N_KV_HEAD, dim=1)
        v = v.repeat_interleave(N_HEAD//N_KV_HEAD, dim=1)
        y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.drop(self.proj(y))

class SwiGLU(nn.Module):
    def __init__(self):
        super().__init__(); hidden = 4*N_EMBD
        self.gate = nn.Linear(N_EMBD, hidden, bias=False)
        self.up = nn.Linear(N_EMBD, hidden, bias=False)
        self.down = nn.Linear(hidden, N_EMBD, bias=False)
    def forward(self,x): return self.down(F.silu(self.gate(x))*self.up(x))

class MoE(nn.Module):
    def __init__(self):
        super().__init__()
        self.router = nn.Linear(N_EMBD, N_EXPERTS, bias=False)
        self.experts = nn.ModuleList([SwiGLU() for _ in range(N_EXPERTS)])
        # Эксперты стартуют как копии одной FFN с малой диверсификацией.
        base = copy.deepcopy(self.experts[0].state_dict())
        for i in range(1,N_EXPERTS):
            self.experts[i].load_state_dict(base)
            with torch.no_grad():
                for p in self.experts[i].parameters(): p.add_(torch.randn_like(p)*0.005)
    def forward(self,x):
        # Sparse dispatch: each expert processes only tokens routed to it.
        # The previous dense-mask version ran every expert over the full batch
        # and sequence, even when most token outputs were discarded.
        logits = self.router(x)
        vals, inds = logits.topk(TOP_K, dim=-1)
        weights = F.softmax(vals, dim=-1)

        flat_x = x.reshape(-1, N_EMBD)
        flat_out = torch.zeros_like(flat_x)
        token_ids = torch.arange(flat_x.shape[0], device=x.device)
        token_ids = token_ids[:, None].expand(-1, TOP_K).reshape(-1)
        expert_ids = inds.reshape(-1)
        route_weights = weights.reshape(-1, 1)

        for expert_id, expert in enumerate(self.experts):
            route_positions = torch.nonzero(
                expert_ids == expert_id, as_tuple=False
            ).flatten()
            if route_positions.numel() == 0:
                continue
            selected_tokens = token_ids.index_select(0, route_positions)
            selected_x = flat_x.index_select(0, selected_tokens)
            selected_weights = route_weights.index_select(0, route_positions)
            expert_output = expert(selected_x)
            flat_out.index_add_(
                0, selected_tokens, expert_output * selected_weights
            )

        return flat_out.view_as(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__(); self.n1=RMSNorm(N_EMBD); self.attn=GQAAttention(); self.n2=RMSNorm(N_EMBD); self.moe=MoE()
    def forward(self,x):
        x = x + self.attn(self.n1(x)); x = x + self.moe(self.n2(x)); return x

class KOTMini(nn.Module):
    def __init__(self):
        super().__init__(); self.token=nn.Embedding(vocab_size,N_EMBD); self.blocks=nn.ModuleList([Block() for _ in range(N_LAYER)]); self.norm=RMSNorm(N_EMBD); self.head=nn.Linear(N_EMBD,vocab_size,bias=False); self.head.weight=self.token.weight
    def forward(self,idx,targets=None):
        x=self.token(idx)
        for block in self.blocks: x=block(x)
        logits=self.head(self.norm(x)); loss=None
        if targets is not None: loss=F.cross_entropy(logits.reshape(-1,vocab_size),targets.reshape(-1))
        return logits,loss
    @torch.no_grad()
    def generate(self,idx,max_new_tokens=350,temperature=0.85,top_k=40):
        self.eval()
        for _ in range(max_new_tokens):
            x=idx[:,-BLOCK_SIZE:]
            logits,_=self(x); logits=logits[:,-1,:]/max(temperature,1e-4)
            if top_k: 
                v,_=torch.topk(logits,min(top_k,logits.size(-1))); logits[logits < v[:,-1,None]]=-float('inf')
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1); idx=torch.cat((idx,nxt),dim=1)
        return idx

model=KOTMini().to(device)
print(f'Параметров: {sum(p.numel() for p in model.parameters()):,}')


## 3. Обучение с управлением весами в реальном времени

Сначала отобразится ползунок, и только потом начнётся обучение. Его можно двигать прямо во время выполнения ячейки. Значение применяется **после каждого optimizer step** к весам четырёх экспертов каждого MoE-блока: `1.00` — без вмешательства, `0.99` — сжать различия на 1% за шаг, `1.01` — увеличить на 1% за шаг. Это накопительное изменение, поэтому даже небольшое отклонение со временем заметно влияет на веса. Каждые 25 шагов обновляются loss/график, каждые 100 — генерация.


In [ ]:
def get_batch(split_name):
    data = train_ids if split_name=='train' else val_ids
    ix = torch.randint(len(data)-BLOCK_SIZE-1,(BATCH_SIZE,))
    x=torch.stack([data[i:i+BLOCK_SIZE] for i in ix]); y=torch.stack([data[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device),y.to(device)

@torch.no_grad()
def estimate_loss():
    model.eval(); vals=[]
    for _ in range(EVAL_BATCHES):
        xb,yb=get_batch('val')
        with torch.autocast('cuda',dtype=torch.float16): _,l=model(xb,yb)
        vals.append(float(l))
    model.train(); return sum(vals)/len(vals)

def change_expert_weights(multiplier):
    # Меняем реальные FFN-параметры: масштабируем отклонение каждого эксперта от среднего.
    with torch.no_grad():
        for li in range(N_LAYER):
            for suffix in ['gate.weight','up.weight','down.weight']:
                ps=[dict(model.named_parameters())[f'blocks.{li}.moe.experts.{ei}.{suffix}'] for ei in range(N_EXPERTS)]
                mean=torch.stack([p.data for p in ps]).mean(0)
                for p in ps: p.data.copy_(mean + multiplier*(p.data-mean))

def make_generation(prompt):
    ids=[stoi.get(c,stoi.get(' ',0)) for c in prompt]
    context=torch.tensor([ids],dtype=torch.long,device=device)
    was_training=model.training
    result=model.generate(context,max_new_tokens=180,temperature=0.85,top_k=40)[0].tolist()
    model.train(was_training)
    return ''.join(itos.get(i,'') for i in result)

optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.1)
scaler=torch.amp.GradScaler('cuda')
history={'step':[],'train':[],'val':[],'multiplier':[]}
prompt_box=widgets.Text(value='Однажды маленький котёнок вышел во двор и увидел',description='Промпт:',layout=widgets.Layout(width='95%'))
weight_slider=widgets.FloatSlider(value=1.0,min=0.90,max=1.10,step=0.001,description='Множитель весов:',readout_format='.3f',continuous_update=True,style={'description_width':'initial'},layout=widgets.Layout(width='95%'))
status=widgets.HTML(value='<b>Подготовка к обучению...</b>')
live_out=widgets.Output()
display(prompt_box,weight_slider,status,live_out)

# Обработчик срабатывает при каждом движении ползунка, пока цикл обучения отдаёт управление UI.
def slider_changed(change):
    if not change or change.get('name') != 'value': return
    alpha=float(change['new'])
    change_expert_weights(alpha)  # немедленно меняем параметры модели
    val_now=estimate_loss()
    with live_out:
        clear_output(wait=True)
        plt.figure(figsize=(8,3)); plt.plot(history['step'],history['train'],label='train')
        if history['val']: plt.plot([x[0] for x in history['val']],[x[1] for x in history['val']],marker='o',label='validation')
        plt.xlabel('Шаг'); plt.ylabel('Loss'); plt.grid(True); plt.legend(); plt.show()
        print(f'ПОЛЗУНОК ИЗМЕНЁН → множитель {alpha:.3f} | текущий validation loss: {val_now:.4f}')
        print('Генерация с изменёнными весами:\\n')
        print(make_generation(prompt_box.value.strip() or 'Однажды в лесу'))

weight_slider.observe(slider_changed,names='value')

best_val=float('inf'); best_state=None; start=time.time(); model.train()
for step in range(1,MAX_STEPS+1):
    xb,yb=get_batch('train'); optimizer.zero_grad(set_to_none=True)
    with torch.autocast('cuda',dtype=torch.float16): _,loss=model(xb,yb)
    scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(optimizer); scaler.update()
    # Читаем актуальное значение виджета на каждом шаге обучения.
    current_multiplier=float(weight_slider.value)
    change_expert_weights(current_multiplier)
    history['step'].append(step); history['train'].append(float(loss.detach())); history['multiplier'].append(current_multiplier)
    if step==1 or step%25==0 or step==MAX_STEPS:
        val_loss=estimate_loss(); history['val'].append((step,val_loss))
        if val_loss < best_val:
            best_val=val_loss; best_state=copy.deepcopy(model.state_dict())
        status.value=(f'<b>Шаг {step}/{MAX_STEPS}</b> | train loss: {float(loss):.4f} | val loss: {val_loss:.4f} | множитель: {current_multiplier:.3f} | лучший val: {best_val:.4f}')
        with live_out:
            clear_output(wait=True)
            plt.figure(figsize=(8,3)); plt.plot(history['step'],history['train'],label='train')
            if history['val']: plt.plot([x[0] for x in history['val']],[x[1] for x in history['val']],marker='o',label='validation')
            plt.xlabel('Шаг'); plt.ylabel('Loss'); plt.grid(True); plt.legend(); plt.show()
            if step%100==0 or step==1 or step==MAX_STEPS:
                print('Генерация при текущих весах:\n')
                print(make_generation(prompt_box.value.strip() or 'Однажды в лесу'))
    await asyncio.sleep(0.01)  # даём Colab обработать движения ползунка во время обучения

final_state=copy.deepcopy(model.state_dict())
torch.save({'model':final_state,'stoi':stoi,'itos':itos},'KOT_V2_Mini_RU_live.pt')
print('Обучение завершено. Лучший validation loss:',round(best_val,4))


## 4. Результаты и сохранение

Во время обучения ты уже видел loss и генерацию. Веса после всех твоих вмешательств сохранены в `final_state`; лучший checkpoint по validation loss — в `best_state`. Для дальнейшей генерации оставь `final_state`, если хочешь сохранить именно свою траекторию эксперимента.

## 4. Сохранение результата

После завершения обучения доступны две версии: `final_state` сохраняет всю траекторию эксперимента с вмешательствами ползунка, а `best_state` — состояние с минимальным validation loss. Следующая ячейка упакует обе версии, словарь и историю обучения в один checkpoint.

In [ ]:
from google.colab import files

checkpoint_path = "KOT_V2_Mini_RU_live.pt"
torch.save(
    {
        "model": final_state,
        "best_model": best_state,
        "stoi": stoi,
        "itos": itos,
        "history": history,
        "config": {
            "n_layer": N_LAYER,
            "n_embd": N_EMBD,
            "n_head": N_HEAD,
            "n_kv_head": N_KV_HEAD,
            "n_experts": N_EXPERTS,
            "top_k": TOP_K,
            "block_size": BLOCK_SIZE,
        },
    },
    checkpoint_path,
)
files.download(checkpoint_path)